# Statistical Analysis: Acceptance Rates and the Partial-Credit Mechanism

Real GPT-6-astra corpus: 4 exercises x 3 rounds x 100 instances = 1,200
correction trajectories, 6,000 graded images (steps 01-05).

This notebook computes the acceptance-rate statistics and the partial-credit
investigation from the current analysis session, using the **same tested
functions** the CLI pipeline uses (`kinematics_grading.analysis.acceptance`),
plus additional descriptive visualizations adapted from
`diffusion_module/diffuser/reverse_corpus_analysis_notebook.ipynb` (the
original corpus-analysis notebook, applied here to embeddings/cluster views
that aren't part of the tested package).

Run from the `diffusion_module_v2/` project root (paths below are relative
to it).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kinematics_grading.analysis.acceptance import (
    acceptance_summary_by_exercise,
    first_accepted_step_distribution,
    partial_credit_on_unsatisfied_rules,
    per_step_pass_rate,
    trajectory_summary,
)
from kinematics_grading.analysis.classification import classify_confusing_robust, confusing_robust_summary
from kinematics_grading.analysis.embeddings import EmbeddingCache
from kinematics_grading.config import DATA_DIR, load_settings
from kinematics_grading.domain import parse_step_info

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

ROOT = Path.cwd()
GRADING_RESULTS = ROOT / "output" / "grading_results"
ANALYSIS_DIR = ROOT / "output" / "analysis"
FIG_DIR = ANALYSIS_DIR / "notebook_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

settings = load_settings()
SCORE_THRESHOLD = settings.analysis.score_threshold
print("SCORE_THRESHOLD:", SCORE_THRESHOLD)

In [ ]:
df = pd.read_json(GRADING_RESULTS / "intermediate_step_rule_grades.jsonl", lines=True)
df["step_order"] = df["step_file"].apply(lambda s: parse_step_info(s).order if parse_step_info(s) else None)
df = df.dropna(subset=["step_order"])
df["step_order"] = df["step_order"].astype(int)

cache = EmbeddingCache(ANALYSIS_DIR / "embeddings_cache.npz")
dists = []
for ex, group in df.groupby("exercise"):
    ref_path = DATA_DIR / f"exercice_{int(ex)}" / f"correct_{int(ex)}.png"
    ref_emb = cache.get(ref_path)
    for _, row in group.iterrows():
        dists.append(EmbeddingCache.cosine_distance(cache.get(Path(row["generated_image"])), ref_emb))
df["dist_to_ref"] = dists

traj_df = trajectory_summary(df, SCORE_THRESHOLD)
print(f"{len(df)} graded images, {len(traj_df)} trajectories")
traj_df.head()

## 1. Acceptance summary by exercise

Overall and per-exercise: what fraction of trajectories are ever accepted (cross 0.60 at some step), and how early/far when they do.

In [ ]:
acc_summary = acceptance_summary_by_exercise(traj_df)
overall_pct = 100.0 * traj_df["has_acceptance"].mean()
print(f"Whole corpus: {traj_df['has_acceptance'].sum()}/{len(traj_df)} accepted ({overall_pct:.2f}%)")
acc_summary.round(2)

## 2. Per-step pass rate (does *this* step, on its own, pass?)

Independent of "first" acceptance: at each step 1-5, what fraction of that exercise's images score >= threshold. This should rise monotonically by construction, but *how* it rises (gradual climb vs. a sudden cliff) is the signal for where the grader starts letting things through.

In [ ]:
step_rates = per_step_pass_rate(df, SCORE_THRESHOLD)

fig, ax = plt.subplots(figsize=(8, 5))
for ex, g in step_rates.groupby("exercise"):
    g = g.sort_values("step_order")
    ax.plot(g["step_order"], g["pct_pass"], marker="o", label=f"Exercise {int(ex)}")
ax.axhline(50, linestyle="--", linewidth=1, color="gray", label="50% of trajectories")
ax.set_xlabel("Correction step")
ax.set_ylabel("% of images passing (score >= 0.60)")
ax.set_title("Per-step pass rate by exercise")
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_ylim(0, 105)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "per_step_pass_rate.png", dpi=150, bbox_inches="tight")
plt.show()

step_rates.pivot(index="step_order", columns="exercise", values="pct_pass").round(1)

### Mean score ratio and mean embedding distance to reference, by step

Descriptive complement (not threshold-based) -- adapted from `reverse_corpus_analysis_notebook.ipynb` cells 8/12/13.

In [ ]:
summary_step = (
    df.groupby(["exercise", "step_order"])
      .agg(mean_score_ratio=("score_ratio", "mean"), mean_dist_ref=("dist_to_ref", "mean"))
      .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ex, g in summary_step.groupby("exercise"):
    g = g.sort_values("step_order")
    axes[0].plot(g["step_order"], g["mean_score_ratio"], marker="o", label=f"Ex {int(ex)}")
    axes[1].plot(g["step_order"], g["mean_dist_ref"], marker="o", label=f"Ex {int(ex)}")
axes[0].axhline(SCORE_THRESHOLD, linestyle="--", linewidth=1, color="gray")
axes[0].set_title("Mean score_ratio by step"); axes[0].set_xlabel("Step"); axes[0].set_xticks([1,2,3,4,5])
axes[1].set_title("Mean embedding distance to reference by step"); axes[1].set_xlabel("Step"); axes[1].set_xticks([1,2,3,4,5])
for ax in axes:
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "mean_score_and_distance_by_step.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Where does each exercise first accept? (the mode)

For each exercise, the % of trajectories whose *first* passing step is each of 1-5, or never. The step with the highest bar is "the step with the highest acceptance rate" for that exercise.

In [ ]:
first_dist = first_accepted_step_distribution(traj_df)
pivot = first_dist.assign(step=first_dist["step"].apply(lambda s: "never" if pd.isna(s) else int(s)))
pivot = pivot.pivot(index="exercise", columns="step", values="pct_of_trajectories")
pivot = pivot[[c for c in [1, 2, 3, 4, 5, "never"] if c in pivot.columns]]

fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot(kind="bar", ax=ax, width=0.8)
ax.set_xlabel("Exercise"); ax.set_ylabel("% of trajectories first accepted here")
ax.set_title("First-accepted-step distribution by exercise")
ax.legend(title="Step", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
fig.savefig(FIG_DIR / "first_accepted_step_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

pivot.round(1)

## 4. Rubric structure -- why the rates differ structurally

Loaded directly from `data/exercice_*/rubric_*.json`.

In [ ]:
import json

rubric_rows = []
for ex in [1, 2, 3, 4]:
    rubric = json.loads((DATA_DIR / f"exercice_{ex}" / f"rubric_{ex}.json").read_text())
    for rule in rubric["rules"]:
        rubric_rows.append({
            "exercise": ex, "rule_id": rule["id"], "quantity": rule["quantity"],
            "relation": rule["relation"], "expected": rule["expected"], "points": rule["points"],
            "max_score": rubric["max_score"],
        })
rubric_df = pd.DataFrame(rubric_rows)
rubric_df

## 5. The partial-credit mechanism -- why early acceptance is possible at all

For every *passing* image, what fraction of its awarded points came from rules the grader itself marked `satisfied: false`. Computed both across all passes and restricted to steps 1-2 (the focus of this investigation).

In [ ]:
partial_all = partial_credit_on_unsatisfied_rules(df, SCORE_THRESHOLD)
partial_early = partial_credit_on_unsatisfied_rules(df[df["step_order"] <= 2], SCORE_THRESHOLD)

comparison = partial_all.merge(partial_early, on="exercise", suffixes=("_all_steps", "_steps_1_2"))

fig, ax = plt.subplots(figsize=(8, 5))
width = 0.35
x = np.arange(len(comparison))
ax.bar(x - width/2, comparison["avg_pct_credit_from_unsatisfied_all_steps"], width, label="All passing steps")
ax.bar(x + width/2, comparison["avg_pct_credit_from_unsatisfied_steps_1_2"], width, label="Steps 1-2 only")
ax.set_xticks(x); ax.set_xticklabels([f"Ex {int(e)}" for e in comparison["exercise"]])
ax.set_ylabel("Avg % of awarded points from unsatisfied rules")
ax.set_title("How much of a passing score rests on partial credit for unmet rules")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "partial_credit_from_unsatisfied_rules.png", dpi=150, bbox_inches="tight")
plt.show()

comparison.round(2)

In [ ]:
# One concrete illustrative example per exercise: an early (step 1-2) pass
# with at least one rule marked unsatisfied.
early_passes = df[(df["step_order"] <= 2) & (df["score_ratio"] >= SCORE_THRESHOLD)]

for ex in [1, 2, 3, 4]:
    sub = early_passes[
        (early_passes["exercise"] == ex)
        & (early_passes["rules"].apply(lambda rs: any(not r["satisfied"] for r in rs)))
    ]
    if sub.empty:
        print(f"Exercise {ex}: no early pass with an unsatisfied rule\n")
        continue
    row = sub.iloc[0]
    print(f"=== Exercise {ex}, step {row['step_order']}, score_ratio={row['score_ratio']:.2f} ===")
    for r in row["rules"]:
        flag = "OK " if r["satisfied"] else "FAIL"
        print(f"  [{flag}] {r['id']:>10s}: {r['awarded']:.1f}/{r['points']:.1f} pts")
    print(f"  feedback: {row['feedback']}\n")

## 6. High-grade-but-far cases (confusing / robust-to-noise)

Adapted from `reverse_corpus_analysis_notebook.ipynb`'s high-grade-far cluster analysis, using the pipeline's own `analysis.embedding_distance_threshold` (fixed) alongside a quantile-based "far" definition (top quartile of distance among passing images, as the original notebook did) for comparison.

In [ ]:
classified = classify_confusing_robust(df, settings.analysis.embedding_distance_threshold, SCORE_THRESHOLD)
cr_summary = confusing_robust_summary(classified)
cr_summary.round(2)

In [ ]:
passing = df[df["score_ratio"] >= SCORE_THRESHOLD]
far_threshold_quantile = passing["dist_to_ref"].quantile(0.75)
print(f"Quantile-based FAR_THRESHOLD (75th pct of dist_to_ref among passing images): {far_threshold_quantile:.4f}")
print(f"Fixed pipeline threshold (analysis.embedding_distance_threshold): {settings.analysis.embedding_distance_threshold}")

fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
for ax, ex in zip(axes.ravel(), [1, 2, 3, 4]):
    g = df[df["exercise"] == ex]
    high_far = g[(g["score_ratio"] >= SCORE_THRESHOLD) & (g["dist_to_ref"] >= far_threshold_quantile)]
    ax.scatter(g["dist_to_ref"], g["score_ratio"], s=10, alpha=0.15, label="All images")
    ax.scatter(high_far["dist_to_ref"], high_far["score_ratio"], s=35, marker="x", color="crimson", label="High grade + far")
    ax.axhline(SCORE_THRESHOLD, linestyle="--", linewidth=0.8, color="gray")
    ax.axvline(far_threshold_quantile, linestyle="--", linewidth=0.8, color="gray")
    ax.set_title(f"Exercise {ex}")
    ax.grid(alpha=0.25)
for ax in axes[-1, :]:
    ax.set_xlabel("Cosine distance to reference")
for ax in axes[:, 0]:
    ax.set_ylabel("score_ratio")
axes[0, 0].legend(loc="lower right", fontsize=8)
fig.suptitle("Score vs. distance-to-reference, high-grade-far cases highlighted", y=1.0)
plt.tight_layout()
fig.savefig(FIG_DIR / "score_vs_distance_high_far.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Embedding-space view (PCA) of first-acceptance points

Adapted from `reverse_corpus_analysis_notebook.ipynb`'s PCA cluster cells: where do first-accepted images sit in ResNet-50 embedding space, relative to the whole corpus, colored by which step first accepted them.

In [ ]:
from sklearn.decomposition import PCA

sample_n = min(4000, len(df))
sample_df = df.sample(sample_n, random_state=42) if len(df) > sample_n else df.copy()
embs = np.stack([cache.get(Path(p)) for p in sample_df["generated_image"]])

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embs)
sample_df = sample_df.assign(pca_1=coords[:, 0], pca_2=coords[:, 1])
print("Explained variance ratio:", pca.explained_variance_ratio_)

first_accept_rows = []
for keys, g in df.groupby(["exercise", "round", "source_exercise", "instance"]):
    g = g.sort_values("step_order")
    hit = g[g["score_ratio"] >= SCORE_THRESHOLD]
    if not hit.empty:
        first_accept_rows.append(hit.iloc[0])
first_accept_df = pd.DataFrame(first_accept_rows)
fa_embs = np.stack([cache.get(Path(p)) for p in first_accept_df["generated_image"]])
fa_coords = pca.transform(fa_embs)
first_accept_df = first_accept_df.assign(pca_1=fa_coords[:, 0], pca_2=fa_coords[:, 1])

step_colors = {1: "#d73027", 2: "#fc8d59", 3: "#fee08b", 4: "#91bfdb", 5: "#4575b4"}
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.scatter(sample_df["pca_1"], sample_df["pca_2"], s=8, alpha=0.08, color="gray", label="All images (sample)")
for step in sorted(first_accept_df["step_order"].unique()):
    g = first_accept_df[first_accept_df["step_order"] == step]
    ax.scatter(g["pca_1"], g["pca_2"], s=35, marker="x", color=step_colors.get(int(step), "black"), label=f"First accepted at step {int(step)}")
ax.set_title("First-acceptance points in ResNet-50 embedding space (PCA)")
ax.set_xlabel("PCA 1"); ax.set_ylabel("PCA 2")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "pca_first_acceptance_points.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

- **Acceptance is near-universal** (98.67% of all 1,200 trajectories eventually pass), but *when* varies sharply by exercise, tracking rubric structure: Exercise 1 (1 coarse rule) and Exercise 2 (3 first-derivative rules) accept early; Exercise 3 (requires 2nd-derivative/curvature correctness) and Exercise 4 (checks raw function value, not slope) hold out until step 3-4.
- **The mechanism enabling early acceptance is partial credit on rules marked unsatisfied.** Across every exercise's step-1/2 passes, the majority have at least one rule the grader itself flags as unmet, and a substantial share of the awarded score (23-91%, worst in Exercise 4) comes from that unmet rule's partial credit alone.
- **This is orthogonal to visual similarity**: the confusing/robust-to-noise breakdown and the high-grade-far scatter plots show the same story from a different angle -- passing scores do not require the image to be visually close to the reference.

This notebook's cells reuse the same tested functions as `kinematics_grading analyze` (`analysis/acceptance.py`), so these numbers will stay in sync with the pipeline as the corpus changes; only the descriptive visualizations (mean-by-step, PCA, high-far scatter) are notebook-only and not part of the tested package.